In [1]:
# Assignment 1: Simple Text Generation API

In [2]:
# Run from the repo root so `app` and `helper_lib` import cleanly
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

/Users/Aditi_1/Documents/Columbia - Fall 2026/Applied Gen AI/genai


In [3]:
from app.bigram_model import BigramModel

corpus = [
    "this is another example sentence",
    "we are generating text based on bigram probabilities",
    "bigram models are simple but effective",
]
model = BigramModel(corpus)
print(model.generate_text("bigram", 6))

bigram probabilities


In [4]:
from app.embedding_model import EmbeddingModel

emb = EmbeddingModel()
r = emb.get_embedding("king")
print(r["dimensions"], r["embedding"][:5])

300 [-0.606440007686615, -0.5120499730110168, 0.006492100190371275, -0.29194000363349915, -0.5651500225067139]


In [5]:
import requests

base = "http://127.0.0.1:8000"
print(requests.post(f"{base}/generate", json={"start_word": "bigram", "length": 5}).json())
print(requests.post(f"{base}/embedding", json={"word": "king"}).json()["dimensions"])

{'generated_text': 'bigram models are generating text'}
300


In [6]:
# Install dependencies (run once). 
!pip install -q -U transformers torch sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def ask(prompt, max_new_tokens=60):
    # Greedy decoding by default -> clean, reproducible answers for demos.
    # (T5 returns only the answer, not prompt + continuation, so no slicing needed.)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True).strip())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cli-base 0.6.0 requires click<8.2, but you have click 8.5.0 which is incompatible.


ModuleNotFoundError: No module named 'transformers'

In [ ]:
# ===============================
# 1. Zero-shot Prompting
# ===============================
print("---- ZERO SHOT ----")
prompt = """ identify if a number is even or odd.
Question: number is: 1
Answer: {"label": "odd"}
Question: number is: 2
Answer: {"label": "even"}
"""
ask("Classify the number as odd or even. Number: 9'")

In [ ]:
# ===============================
# 2. One-shot Prompting
# ===============================
print("---- ONE SHOT ----")
prompt = """Classify the topic of these sentences (topics: sports, politics, tech).

Sentence: The game was terrible.
Sentiment: Sports

Sentence: I really disliked the company.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 3. Few-shot Prompting
# ===============================
prompt = """Extract the names from these sentences.

Sentence: Mathew loves pizza.
Sentiment: Mathew

Sentence: While building a bike, Abi injured her toe.
Sentiment: Abi

Sentence: Between the auditorium and the dance floor Kelly got lost.
Sentiment:"""
ask(prompt)

In [ ]:
# ===============================
# 4. Chain-of-Thought Prompting
# (Model is guided to reason step by step)
# ===============================
print("---- CHAIN OF THOUGHT ----")
prompt = """Question: If a train travels 60km in 1 hour, how far can it travel in 3 hours?
Let's think step by step."""
ask(prompt)

In [ ]:
# ===============================
# 5. Role Prompting (persona: junior school tutor)
# ===============================
print("---- ROLE PROMPT: junior school tutor ----")
prompt = """You are the school teacher for a classroom of 7 year old students.
Explain the moon to these 7 year old students in simple words."""
ask(prompt, max_new_tokens=80)